In [6]:
import sys
from pathlib import Path
import pickle
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
CALIBRATION_DIR = ROOT_DIR / "data/calibration_results"


# 1. Stereo Matching

In [7]:
# Rectificar todo
# Cargar los mapas de rectificación precalculados
with (CALIBRATION_DIR / "stereo_maps.pkl").open("rb") as f:
    maps: dict = pickle.load(f)
map1x = maps['left_map_x']
map1y = maps['left_map_y']
map2x = maps['right_map_x']
map2y = maps['right_map_y']

# Configurar los parámetros de StereoSGBM
block_size = 11
stereo = cv.StereoSGBM_create(
    minDisparity=0,
    numDisparities=240,
    blockSize=block_size,
    P1=8 * 1 * block_size**2,
    P2=32 * 1 * block_size**2,
    disp12MaxDiff=0,
    uniquenessRatio=5,
    speckleWindowSize=200,
    speckleRange=2,
    preFilterCap=0,
    mode=cv.STEREO_SGBM_MODE_SGBM
)

# Definir rutas del dataset
dataset_path = '../data/raw/palacio/'
images_left = sorted(glob.glob(os.path.join(dataset_path, 'left*.jpg'))) 
images_right = sorted(glob.glob(os.path.join(dataset_path, 'right*.jpg')))

output_dir = '../data/processed/disparity/'
os.makedirs(output_dir, exist_ok=True)

disparity_maps = []

# # Bucle principal de rectificación y disparidad
# for img_left_path, img_right_path in zip(images_left, images_right):
    
#     img_left = cv.imread(img_left_path, cv.IMREAD_GRAYSCALE)
#     img_right = cv.imread(img_right_path, cv.IMREAD_GRAYSCALE)
    
#     if img_left is None or img_right is None:
#         continue

#     # Aplicar los mapas cargados de tu .pkl
#     rect_left = cv.remap(img_left, map1x, map1y, cv.INTER_LINEAR)
#     rect_right = cv.remap(img_right, map2x, map2y, cv.INTER_LINEAR)

#     # Calcular disparidad
#     disparity_16 = stereo.compute(rect_left, rect_right)
#     disparity = disparity_16.astype(np.float32) / 16.0
    
#     disparity_maps.append(disparity)
    
#     # Guardar imagen para visualización/informe
#     base_name = os.path.basename(img_left_path).replace('left', 'disp').replace('.jpg', '.png')
#     disparity_vis = cv.normalize(disparity, None, alpha=0, beta=255, norm_type=cv.NORM_MINMAX, dtype=cv.CV_8U)
#     cv.imwrite(os.path.join(output_dir, base_name), disparity_vis)

# print(f"Procesados {len(disparity_maps)} pares. Mapas guardados en {output_dir}")

# 2. Computar Nube local

In [8]:
import cv2 as cv
import numpy as np
import open3d as o3d
import pickle
import glob
import os
from pathlib import Path

# Cargar parámetros de calibración para obtener la matriz Q
with open('../data/calibration_results/stereo_calibration.pkl', 'rb') as f:
    calib = pickle.load(f)

K1, D1 = calib['left_K'], calib['left_dist']
K2, D2 = calib['right_K'], calib['right_dist']
R, T = calib['R'], calib['T']
image_size = calib['image_size']

# cv.stereoRectify devuelve varias matrices, la quinta es Q
_, _, _, _, Q, _, _ = cv.stereoRectify(K1, D1, K2, D2, image_size, R, T)

dataset_path = '../data/raw/palacio/'
disp_path = '../data/processed/disparity/'

images_left = sorted(glob.glob(os.path.join(dataset_path, '*left*.jpg')))
disparity_files = sorted(glob.glob(os.path.join(disp_path, '*.png'))) # Asumiendo que guardaste .npy o .png

# Crear carpeta para las nubes locales
output_pcd_dir = '../data/processed/local_clouds/'
os.makedirs(output_pcd_dir, exist_ok=True)

# Cargar también los mapas de rectificación para obtener el color correcto de los puntos
with open("../data/calibration_results/stereo_maps.pkl", "rb") as f:
    maps = pickle.load(f)
map1x, map1y = maps['left_map_x'], maps['left_map_y']

# Bucle para computar la nube local de cada par
for img_left_path, disp_path in zip(images_left, disparity_files):
    
    # Leer imagen original izquierda y mapa de disparidad
    img_left = cv.imread(img_left_path, cv.IMREAD_COLOR)
    
    # IMPORTANTE: Si guardaste la disparidad como imagen (.png) normalizada de 0-255, 
    # perdiste la escala real. Deberías usar la matriz original flotante. 
    # Asumiendo que las tienes en memoria o las recalculas rápidamente:
    
    # Recalculamos rápido la disparidad real (si no guardaste los arrays en .npy)
    img_left_gray = cv.cvtColor(img_left, cv.COLOR_BGR2GRAY)
    img_right_gray = cv.imread(img_left_path.replace('left', 'right'), cv.IMREAD_GRAYSCALE)
    rect_left = cv.remap(img_left_gray, map1x, map1y, cv.INTER_LINEAR)
    rect_right = cv.remap(img_right_gray, maps['right_map_x'], maps['right_map_y'], cv.INTER_LINEAR)
    
    # Usar el mismo objeto stereo que configuraste antes
    disparity_16 = stereo.compute(rect_left, rect_right)
    disparity = disparity_16.astype(np.float32) / 16.0
    
    # Reproyectar a 3D usando Q
    points_3D = cv.reprojectImageTo3D(disparity, Q)
    
    # Obtener los colores rectificando la imagen izquierda a color
    rect_left_color = cv.remap(img_left, map1x, map1y, cv.INTER_LINEAR)
    colors = cv.cvtColor(rect_left_color, cv.COLOR_BGR2RGB).astype(np.float32) / 255.0
    
    # Filtrar puntos inválidos (donde la disparidad falló o es muy pequeña)
    # Valores de disparidad <= 0 suelen ser ruido de fondo o píxeles sin correspondencia
    mask = disparity > 0
    valid_points = points_3D[mask]
    valid_colors = colors[mask]
    
    # Crear objeto PointCloud de Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(valid_points)
    pcd.colors = o3d.utility.Vector3dVector(valid_colors)
    
    # Guardar la nube local
    base_name = os.path.basename(img_left_path).replace('left', 'local_cloud').replace('.jpg', '.ply')
    o3d.io.write_point_cloud(os.path.join(output_pcd_dir, base_name), pcd)

print(f"Nubes locales generadas y guardadas en {output_pcd_dir}")

Nubes locales generadas y guardadas en ../data/processed/local_clouds/


# 3. Pose global

In [ ]:
# import cv2 as cv
# import numpy as np
# import open3d as o3d

# # 1. Preparar los puntos 3D teóricos del tablero (el mismo 'objp' que usaste en la calibración)
# # Asegúrate de usar los valores correctos para tu dataset (ej. 9x6 esquinas, 28mm)
# columnas = 9
# filas = 6
# tamanio_cuadrado_mm = 28.0

# objp = np.zeros((filas * columnas, 3), np.float32)
# objp[:, :2] = np.mgrid[0:columnas, 0:filas].T.reshape(-1, 2) * tamanio_cuadrado_mm

# # 2. Obtener la matriz intrínseca de la cámara rectificada
# # P1 es la matriz de proyección devuelta por cv.stereoRectify para la cámara izquierda.
# # Sus primeros 3x3 componentes forman la nueva matriz intrínseca.
# _, _, P1, _, _, _, _ = cv.stereoRectify(K1, D1, K2, D2, image_size, R, T)
# K_rect = P1[:3, :3]
# dist_rect = np.zeros(4) # Las imágenes rectificadas ya no tienen distorsión

# # Dentro de tu bucle for por cada imagen...
# # Asumimos que 'rect_left' es la imagen izquierda rectificada (en escala de grises)
# # y 'pcd' es la PointCloud local que generaste en el paso anterior.

# ret, corners = cv.findChessboardCorners(rect_left, (columnas, filas), None)

# if ret:
#     # Opcional pero recomendado: refinar las esquinas a nivel subpíxel
#     corners = cv.cornerSubPix(rect_left, corners, (11, 11), (-1, -1), criteria)
    
#     # 3. Estimar la pose (de mundo a cámara)
#     ret_pnp, rvec, tvec = cv.solvePnP(objp, corners, K_rect, dist_rect)
    
#     if ret_pnp:
#         # cv.solvePnP nos da el vector de rotación (rvec). 
#         # cv.Rodrigues lo convierte a una matriz de rotación 3x3.
#         R_cam, _ = cv.Rodrigues(rvec)
        
#         # rvec y tvec transforman puntos del MUNDO a la CÁMARA (X_cam = R_cam * X_mundo + tvec)
#         # Nosotros tenemos la nube en la CÁMARA y queremos llevarla al MUNDO.
#         # Por lo tanto, necesitamos invertir la transformación.
#         R_inv = R_cam.T
#         t_inv = -R_inv @ tvec
        
#         # 4. Crear la matriz de transformación homogénea 4x4
#         T_global = np.eye(4)
#         T_global[:3, :3] = R_inv
#         T_global[:3, 3] = t_inv.flatten()
        
#         # 5. Aplicar la transformación a la nube de puntos local en Open3D
#         pcd.transform(T_global)
        
#         # (Opcional) Guardar la nube global para este par
#         # o3d.io.write_point_cloud("nube_global_par.ply", pcd)
# else:
#     print("No se detectó el tablero de referencia en esta vista.")

No se detectó el tablero de referencia en esta vista.


# 4. Transformación